# Task 2: Laning & Overtaking with SB3 PPO

This notebook trains a PPO agent on the newer `highway-env` / Stable-Baselines3 API, following the older Task 2 setup from `racetrack-agents` where three slower non-agent vehicles are spawned and the ego vehicle must lane-follow while overtaking.

The older DQN command used `--spawn_vehicles 3`, `--batch_size 256`, `--lr 0.00005`, `--lr_decay`, `--arch Identity`, and `--fc_layers 3`. The cells below map those ideas to SB3 PPO with a 3-layer MLP policy, linear learning-rate decay, and `other_vehicles=3` in the `racetrack-oval-v0` config.

In [12]:
# If this notebook is running in a fresh environment, install the core packages first.
# In the local repo environment you can usually leave this cell commented out.
#
# %pip install "highway-env>=1.8" "stable-baselines3[extra]>=2.0" tensorboard moviepy

from pathlib import Path
from copy import deepcopy
import base64
import os
import platform
import random

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import highway_env  # Registers highway-env environments in many versions.
import numpy as np
import torch

from IPython.display import HTML, display
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from highway_env.envs.racetrack_env import RacetrackEnvOval

# Newer gymnasium versions can register an external environment package explicitly.
# Older highway-env versions register on import, so we keep this tolerant.
try:
    gym.register_envs(highway_env)
except Exception:
    pass


## Experiment Config

Task 2 is represented by `other_vehicles=3`. The notebook now starts from `RacetrackEnvOval.default_config()` and overrides only the pieces that define this experiment, so future highway-env API changes are easier to absorb.

In [13]:
SEED = 42
ENV_ID = "racetrack-v0"

# Keep full training as the default. For an end-to-end notebook smoke test, run
# `FAST_DEV_RUN=1` in the process environment before executing the notebook.
# FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "0") == "1"
FAST_DEV_RUN = False
EXP_ID = "sb3_ppo_task2_laning_overtaking_fastdev" if FAST_DEV_RUN else "sb3_ppo_task2_laning_overtaking"

CWD = Path.cwd()
if (CWD / "racetrack_env.py").exists():
    WORK_DIR = CWD
elif (CWD / "racetrack-agents").exists():
    WORK_DIR = CWD / "racetrack-agents"
else:
    WORK_DIR = CWD

RUN_DIR = WORK_DIR / "runs" / EXP_ID
MODEL_DIR = RUN_DIR / "models"
BEST_MODEL_DIR = MODEL_DIR / "best"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
VIDEO_DIR = RUN_DIR / "videos"

for directory in [MODEL_DIR, BEST_MODEL_DIR, CHECKPOINT_DIR, LOG_DIR, VIDEO_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Reproducibility: exact runs can still vary across machines/GPU kernels.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Mapping from the older DQN command:
#   --spawn_vehicles 3  -> config["other_vehicles"] = 3
#   --batch_size 256    -> PPO minibatch size 256
#   --lr 0.00005        -> PPO starting learning rate 5e-5
#   --lr_decay          -> SB3 linear learning-rate schedule
#   --arch Identity     -> flatten the occupancy grid, then use an MLP
#   --fc_layers 3       -> 3 hidden layers in the actor and critic

USE_RICH_OCCUPANCY_FEATURES = False

def racetrack_oval_default_config():
    """Return the official highway-env oval racetrack defaults.

    The installed highway-env version in this conda env exposes
    RacetrackEnvOval.default_config as an unbound method, while newer versions
    may expose it as a classmethod. This helper supports both forms.
    """
    try:
        config = RacetrackEnvOval.default_config()
    except TypeError:
        config = RacetrackEnvOval.default_config(RacetrackEnvOval)
    return deepcopy(config)


def make_task2_config():
    """Start from highway-env's built-in RacetrackEnvOval config and override only what this experiment needs."""
    config = racetrack_oval_default_config()

    # Task 2: match older `--spawn_vehicles 3` with three slower IDM traffic vehicles.
    config["other_vehicles"] = 3

    # Keep the API's oval geometry fixed; `0` would randomize straight length across episodes.
    config["length"] = 100

    # Keep three lanes so the policy can learn both lane keeping and overtaking choices.
    config["no_lanes"] = 3

    # Stop episodes when the agent leaves the road; this gives PPO a clearer failure signal.
    config["terminate_off_road"] = True

    # Larger renders make the exported best-episode video easier to inspect.
    config["screen_width"] = 1000

    # Keep render output square so the oval track is not visually stretched.
    config["screen_height"] = 1000

    # Optional: use the richer observation stack from sb3_racetrack_oval_ppo.py.
    # The default two channels (`presence`, `on_road`) better match the older Identity DQN/PPO setup.
    if USE_RICH_OCCUPANCY_FEATURES:
        config["observation"]["features"] = [
            "presence",  # Whether a vehicle occupies a grid cell.
            "on_road",   # Road mask; helps the policy identify valid track cells.
            "x",         # Relative longitudinal position in the occupancy grid.
            "y",         # Relative lateral position in the occupancy grid.
            "vx",        # Longitudinal velocity of observed vehicles.
            "vy",        # Lateral velocity of observed vehicles.
            "cos_h",     # Cosine of heading for directional context.
            "sin_h",     # Sine of heading for directional context.
            "long_off",  # Longitudinal lane offset, useful near curves.
            "lat_off",   # Lateral lane offset, useful for lane centering.
            "ang_off",   # Angular lane offset, useful for steering through turns.
        ]

    return config


TASK2_CONFIG = make_task2_config()

# Fast mode now performs a real short PPO run with multiple rollouts/evaluations so progress is visible.
FAST_DEV_TIMESTEPS = int(os.environ.get("FAST_DEV_TIMESTEPS", "4096"))

# Full mode keeps the older command's 5000-episode intent, using the current API's episode horizon.
N_EPISODES = 5000
TOTAL_TIMESTEPS = FAST_DEV_TIMESTEPS if FAST_DEV_RUN else N_EPISODES * TASK2_CONFIG["duration"]

# Use one env in notebooks on Windows; full mode can use several vectorized workers.
N_ENVS = 1 if FAST_DEV_RUN else min(8, max(1, os.cpu_count() or 1))

# PPO minibatch size: smaller in fast mode, older command value in full mode.
BATCH_SIZE = 64 if FAST_DEV_RUN else 256

# Rollout length per env before each PPO update; fast mode still collects meaningful trajectories.
N_STEPS = 512 if FAST_DEV_RUN else 2048

# Starting learning rate, mapped from the older `--lr 0.00005` setting.
LEARNING_RATE = 5e-5

# Number of SGD passes per PPO update; fewer in fast mode keeps iteration time reasonable.
N_EPOCHS = 4 if FAST_DEV_RUN else 10

# Evaluation cadence in environment steps; fast mode evaluates often so you can see progress.
EVAL_FREQ = 1024 if FAST_DEV_RUN else max(10_000 // N_ENVS, 1)

# Evaluation episodes per callback; two is enough to see trend during a quick run.
N_EVAL_EPISODES = 2 if FAST_DEV_RUN else 5


## Build Training and Evaluation Environments

SB3 trains on vectorized environments. `DummyVecEnv` is the safest default inside notebooks on Windows. For longer command-line runs, set `USE_SUBPROC = True`.

In [14]:
USE_SUBPROC = False

def make_task2_env(render_mode=None):
    """Create one Task 2 racetrack environment."""
    return gym.make(ENV_ID, config=TASK2_CONFIG, render_mode=render_mode)

vec_env_cls = SubprocVecEnv if USE_SUBPROC and N_ENVS > 1 else DummyVecEnv

train_env = make_vec_env(
    lambda: make_task2_env(),
    n_envs=N_ENVS,
    seed=SEED,
    vec_env_cls=vec_env_cls,
)
# Evaluation stays single-env so callback results are easy to interpret.
eval_env = make_vec_env(
    lambda: make_task2_env(),
    n_envs=1,
    seed=SEED + 10_000,
    vec_env_cls=DummyVecEnv,
)


## Define PPO

The PPO policy uses a 3-layer actor and critic MLP, matching the spirit of `--arch Identity --fc_layers 3` from the older code: flatten the occupancy grid, then learn dense policy/value heads. Comments below explain every model setting that differs from SB3 defaults or maps to the older command.

In [15]:
def linear_schedule(initial_value):
    """SB3 schedule: progress_remaining moves from 1.0 to 0.0 during training."""
    def schedule(progress_remaining):
        return progress_remaining * initial_value
    return schedule

policy_kwargs = {
    # Tanh matches the older TensorFlow PPO hidden-layer style and works well with normalized features.
    "activation_fn": torch.nn.Tanh,

    # Three 256-unit layers map the old `--fc_layers 3` / default `--fc_width 256` idea to SB3.
    "net_arch": {
        "pi": [256, 256, 256],  # Actor network: outputs the lateral continuous-control distribution.
        "vf": [256, 256, 256],  # Critic network: estimates state value for PPO advantage learning.
    },
}

model = PPO(
    policy="MlpPolicy",  # Flattened occupancy-grid input, equivalent in spirit to the older Identity backbone.
    env=train_env,  # Vectorized Task 2 racetrack environment.
    learning_rate=linear_schedule(LEARNING_RATE),  # Implements the older `--lr_decay` behavior.
    n_steps=N_STEPS,  # Rollout length before each PPO update.
    batch_size=BATCH_SIZE,  # Minibatch size for PPO optimization.
    n_epochs=N_EPOCHS,  # Number of optimization passes over each rollout buffer.
    gamma=0.90,  # Discount factor, matching the older PPO default `gae_gamma`.
    gae_lambda=0.95,  # GAE smoothing, matching the older PPO default.
    clip_range=0.20,  # PPO clipping epsilon, matching the older `ppo_epsilon` default.
    ent_coef=0.001,  # Small entropy bonus to keep exploration alive during overtaking.
    max_grad_norm=0.5,  # Gradient clipping for stable policy updates.
    policy_kwargs=policy_kwargs,  # Actor/critic architecture defined above.
    tensorboard_log=str(LOG_DIR),  # Training curves and eval metrics.
    seed=SEED,  # Reproducible initialization and rollout seeds where supported.
    verbose=1,  # Print rollout/evaluation progress in notebook output.
)

model.policy


Using cuda device


ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=288, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=288, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): Tanh()
    )
  )
  (action_net): Linear(in_features=256, out_features=1, bias=True)
  (value_net): Linear(in_f

## Train and Save the Best Model

`EvalCallback` periodically runs deterministic evaluations and writes the best model to disk. TensorBoard logs are stored under `runs/sb3_ppo_task2_laning_overtaking/logs`.

In [16]:
# Uncomment these lines in an interactive notebook session.
# %load_ext tensorboard
# %tensorboard --logdir runs/sb3_ppo_task2_laning_overtaking/logs

In [ ]:
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=str(BEST_MODEL_DIR),
    log_path=str(LOG_DIR / "eval"),
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=True,
    render=False,
)

checkpoint_callback = CheckpointCallback(
    save_freq=max(50_000 // N_ENVS, 1),
    save_path=str(CHECKPOINT_DIR),
    name_prefix="ppo_task2_checkpoint",
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[eval_callback, checkpoint_callback],
    tb_log_name="PPO_task2",
)

model.save(MODEL_DIR / "ppo_task2_last")
train_env.close()
eval_env.close()


Logging to c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_task2_laning_overtaking\logs\PPO_task2_6
Eval num_timesteps=10000, episode_reward=17.39 +/- 5.21
Episode length: 20.40 +/- 5.95
---------------------------------
| eval/              |          |
|    mean_ep_length  | 20.4     |
|    mean_reward     | 17.4     |
| time/              |          |
|    total_timesteps | 10000    |
---------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 8.74     |
|    ep_rew_mean     | 4.41     |
| time/              |          |
|    fps             | 73       |
|    iterations      | 1        |
|    time_elapsed    | 223      |
|    total_timesteps | 16384    |
---------------------------------
Eval num_timesteps=20000, episode_reward=33.70 +/- 10.99
Episode length: 54.40 +/- 17.32
-----------------------------------------
| eval/                   |             |
|    mean_ep_length       |

## Load the Best PPO Checkpoint

If training was interrupted before an evaluation improved, fall back to the last saved model.

In [ ]:
best_model_path = BEST_MODEL_DIR / "best_model.zip"
last_model_path = MODEL_DIR / "ppo_task2_last.zip"

if best_model_path.exists():
    trained_model = PPO.load(best_model_path)
    print(f"Loaded best model: {best_model_path}")
else:
    trained_model = PPO.load(last_model_path)
    print(f"Best model was not found, loaded last model: {last_model_path}")


Loaded best model: c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_task2_laning_overtaking_fastdev\models\best\best_model.zip


## Find and Export the Best Evaluation Episode

The first pass evaluates deterministic rollouts over fixed seeds without recording. The best seed is then replayed once with `RecordVideo`, producing a single video for the strongest episode found in this sweep.

In [ ]:
def run_episode(model, seed, record=False, name_prefix="ppo_task2_best_episode"):
    """Run one deterministic episode and optionally record it to VIDEO_DIR."""
    render_mode = "rgb_array" if record else None
    env = gym.make(ENV_ID, config=TASK2_CONFIG, render_mode=render_mode)

    if record:
        env = RecordVideo(
            env,
            video_folder=str(VIDEO_DIR),
            name_prefix=name_prefix,
            episode_trigger=lambda episode_id: episode_id == 0,
        )
    obs, info = env.reset(seed=seed)
    done = False
    truncated = False
    total_reward = 0.0
    episode_length = 0

    while not (done or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        total_reward += float(reward)
        episode_length += 1
        if record:
            env.render()

    env.close()
    return total_reward, episode_length

candidate_seeds = list(range(SEED, SEED + (1 if FAST_DEV_RUN else 25)))
episode_scores = []

for seed in candidate_seeds:
    reward, length = run_episode(trained_model, seed=seed, record=False)
    episode_scores.append({"seed": seed, "reward": reward, "length": length})

best_episode = max(episode_scores, key=lambda item: item["reward"])
best_episode


{'seed': 42, 'reward': 43.69442894050416, 'length': 71}

In [ ]:
video_prefix = f"ppo_task2_best_seed_{best_episode['seed']}"
recorded_reward, recorded_length = run_episode(
    trained_model,
    seed=best_episode["seed"],
    record=True,
    name_prefix=video_prefix,
)

video_files = sorted(VIDEO_DIR.glob(f"{video_prefix}*.mp4"), key=lambda path: path.stat().st_mtime)
best_video_path = video_files[-1] if video_files else None

print(f"Recorded reward: {recorded_reward:.3f}")
print(f"Recorded length: {recorded_length}")
print(f"Video path: {best_video_path}")


c:\Users\16469\anaconda3\envs\circuit\lib\site-packages\gymnasium\wrappers\record_video.py:94: UserWarning: WARN: Overwriting existing videos at c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_task2_laning_overtaking_fastdev\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


MoviePy - Building video c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_task2_laning_overtaking_fastdev\videos\ppo_task2_best_seed_42-episode-0.mp4.
MoviePy - Writing video c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_task2_laning_overtaking_fastdev\videos\ppo_task2_best_seed_42-episode-0.mp4



MoviePy - Done !
MoviePy - video ready c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_task2_laning_overtaking_fastdev\videos\ppo_task2_best_seed_42-episode-0.mp4
Recorded reward: 43.694
Recorded length: 71
Video path: c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_task2_laning_overtaking_fastdev\videos\ppo_task2_best_seed_42-episode-0.mp4


## Display the Exported Video

In [ ]:
def show_video(video_path, width=720):
    """Embed an exported mp4 directly in the notebook."""
    video_path = Path(video_path)
    video_bytes = video_path.read_bytes()
    encoded = base64.b64encode(video_bytes).decode("ascii")
    display(HTML(f"""
    <video width="{width}" controls>
      <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
    </video>
    """))

if best_video_path is not None:
    show_video(best_video_path)
else:
    print("No video file was found. Check that moviepy/ffmpeg are installed and rerun the recording cell.")


## Optional: TensorBoard

Run this cell while training or after training to inspect reward, loss, entropy, KL, and evaluation curves.

In [ ]:
# Uncomment these lines in an interactive notebook session.

